# Heavy-Tailed ABMs: Cont-Bouchaud & OFC

This notebook reproduces power-law experiments from two agent-based models:

1. **Cont-Bouchaud (CB)**: herd behaviour and fat tails in financial markets.
2. **Olami-Feder-Christensen (OFC)**: self-organised criticality in seismic faults.

Sections:
1. Setup & imports
2. CB simulation demo
3. CB phase diagram
4. CB calibration (MLE + ABC)
5. OFC simulation demo
6. OFC phase diagram
7. OFC calibration (MLE + ABC)

## 1. Setup & imports

In [1]:
import os
import sys
import warnings

warnings.filterwarnings('ignore')

# Ensure the project root is on the path
PROJECT_ROOT = os.path.abspath(os.path.dirname('.'))
if PROJECT_ROOT not in sys.path:
    sys.path.insert(0, PROJECT_ROOT)

import numpy as np
import matplotlib
matplotlib.use('Agg')  # headless backend; switch to 'inline' in Jupyter
import matplotlib.pyplot as plt
import matplotlib.image as mpimg
from IPython.display import display, Image

from models.cont_bouchaud import simulate_cb
from models.ofc import simulate_ofc
from utils.powerlaw_fit import fit_powerlaw, gutenberg_richter_b
from utils.plotting import (
    plot_ccdf, plot_return_series,
    plot_cb_demo_ccdf,
    plot_ofc_demo_timeseries, plot_ofc_demo_ccdf,
    FIGURES_DIR, _apply_style, _ensure_figures_dir)

_ensure_figures_dir()
print('Setup complete. Figures will be saved to:', FIGURES_DIR)

Setup complete. Figures will be saved to: /Users/gabriel/Desktop/heavy_tails_abm/figures


## 2. Cont-Bouchaud simulation demo

Single run at `p=0.59` (near the percolation threshold), `L=64`, 10 000 steps.

In [ ]:
# CB single run
CB_L, CB_P, CB_A, CB_STEPS = 64, 0.59, 0.01, 10_000
returns_cb = simulate_cb(CB_L, CB_P, CB_A, CB_STEPS, seed=42)

print(f'CB returns: mean={returns_cb.mean():.4f}, std={returns_cb.std():.4f}')
print(f'Kurtosis: {float(np.mean((returns_cb - returns_cb.mean())**4) / returns_cb.std()**4):.2f}')

In [ ]:
# Return time series
save_ts = os.path.join(FIGURES_DIR, 'cb_demo_timeseries.pdf')
plot_return_series(returns_cb, title='CB returns (p=0.59, L=64)', save_path=save_ts)
print('Saved:', save_ts)

In [ ]:
# CCDF of absolute returns
abs_ret = np.abs(returns_cb[returns_cb != 0])
fit_res = fit_powerlaw(abs_ret)
print(f"Power-law alpha = {fit_res['alpha']:.3f} (xmin={fit_res['xmin']:.4f})")
print(f"vs lognormal: R={fit_res['R_lognormal']:.3f}, p={fit_res['p_lognormal']:.3f}")

save_ccdf = os.path.join(FIGURES_DIR, 'cb_demo_ccdf.pdf')
plot_cb_demo_ccdf(abs_ret, fit_res, CB_P, CB_L, save_ccdf)
print('Saved:', save_ccdf)

## 3. CB Phase Diagram

Grid search over `p ∈ [0.40, 0.70]` and `L ∈ {32, 64, 128}`.

In [ ]:
from experiments.cb_phase_diagram import (
    run_cb_phase_diagram, plot_cb_phase_diagram,
    P_GRID, L_VALUES as CB_L_VALUES)

cb_exponents = run_cb_phase_diagram()
plot_cb_phase_diagram(cb_exponents)

## 4. CB Calibration on CAC 40 and S&P 500

### 4a. MLE calibration

In [ ]:
from data.download_data import download_stock_returns
from experiments.cb_calibration import run_mle_calibration as cb_mle

print('Downloading stock returns ...')
returns_dict = download_stock_returns(
    tickers=['^FCHI', '^GSPC'],
    start='2000-01-01',
    end='2024-01-01')

for ticker, rets in returns_dict.items():
    abs_ret = np.abs(rets.dropna().values)
    abs_ret = abs_ret[abs_ret > 0]
    res = fit_powerlaw(abs_ret)
    alpha_emp = res['alpha']
    print(f'\n{ticker}: empirical alpha = {alpha_emp:.4f}')
    cb_mle(alpha_emp, ticker)

### 4b. ABC calibration

In [ ]:
from experiments.cb_calibration import run_abc_calibration as cb_abc

for ticker, rets in returns_dict.items():
    abs_ret = np.abs(rets.dropna().values)
    abs_ret = abs_ret[abs_ret > 0]
    res = fit_powerlaw(abs_ret)
    alpha_emp = res['alpha']
    print(f'\nABC calibration for {ticker} (alpha_emp={alpha_emp:.4f}) ...')
    try:
        cb_abc(alpha_emp, ticker)
    except Exception as e:
        print(f'  Skipped: {e}')

## 5. OFC Simulation Demo

Single run with `alpha_ofc=0.20`, `L=64`, `n_events=20_000`, `seed=42`.

In [5]:
# OFC demo parameters (fast: <10 s)
OFC_L      = 64
OFC_ALPHA  = 0.20
OFC_EVENTS = 20_000

sizes_ofc = simulate_ofc(OFC_L, OFC_ALPHA, OFC_EVENTS, seed=42)

print(f"Avalanche sizes: min={sizes_ofc.min()}, max={sizes_ofc.max()}")
print(f"Mean size: {sizes_ofc.mean():.2f}")
print(f"Fraction size-1: {(sizes_ofc == 1).mean():.3f}")

save_ts_ofc = os.path.join(FIGURES_DIR, "ofc_demo_timeseries.pdf")
plot_ofc_demo_timeseries(sizes_ofc, OFC_L, OFC_ALPHA, save_ts_ofc)
print("Saved:", save_ts_ofc)

Avalanche sizes: min=0, max=65
Mean size: 0.45
Fraction size-1: 0.080


INFO:fontTools.subset:maxp pruned
INFO:fontTools.subset:cmap pruned
INFO:fontTools.subset:kern dropped
INFO:fontTools.subset:post pruned
INFO:fontTools.subset:FFTM dropped
INFO:fontTools.subset:GPOS pruned
INFO:fontTools.subset:GSUB pruned
INFO:fontTools.subset:glyf pruned
INFO:fontTools.subset:Added gid0 to subset
INFO:fontTools.subset:Added first four glyphs to subset
INFO:fontTools.subset:Closing glyph list over 'MATH': 37 glyphs before
INFO:fontTools.subset:Glyph names: ['.notdef', '.null', 'A', 'C', 'E', 'F', 'L', 'O', 'a', 'c', 'comma', 'd', 'e', 'equal', 'five', 'four', 'h', 'i', 'l', 'm', 'n', 'nonmarkingreturn', 'one', 'p', 'parenleft', 'parenright', 'period', 'r', 's', 'six', 'space', 't', 'two', 'v', 'x', 'z', 'zero']
INFO:fontTools.subset:Glyph IDs:   [0, 1, 2, 3, 11, 12, 15, 17, 19, 20, 21, 23, 24, 25, 32, 36, 38, 40, 41, 47, 50, 68, 70, 71, 72, 75, 76, 79, 80, 81, 83, 85, 86, 87, 89, 91, 93]
INFO:fontTools.subset:Closed glyph list over 'MATH': 43 glyphs after
INFO:fontToo

Saved: /Users/gabriel/Desktop/heavy_tails_abm/figures/ofc_demo_timeseries.pdf


In [6]:
# CCDF with power-law fit overlay
fit_res_ofc = fit_powerlaw(sizes_ofc.astype(float))
print(f"Power-law alpha = {fit_res_ofc['alpha']:.3f}  (xmin={fit_res_ofc['xmin']:.1f})")

mags_ofc = np.log10(sizes_ofc[sizes_ofc > 0].astype(float))
b_ofc = gutenberg_richter_b(mags_ofc)
print(f"G-R b-value = {b_ofc:.4f}")

save_ccdf_ofc = os.path.join(FIGURES_DIR, "ofc_demo_ccdf.pdf")
plot_ofc_demo_ccdf(sizes_ofc, fit_res_ofc, OFC_L, OFC_ALPHA, save_ccdf_ofc)
print("Saved:", save_ccdf_ofc)

Power-law alpha = 2.591  (xmin=4.0)
G-R b-value = 1.5635


INFO:fontTools.subset:maxp pruned
INFO:fontTools.subset:cmap pruned
INFO:fontTools.subset:kern dropped
INFO:fontTools.subset:post pruned
INFO:fontTools.subset:FFTM dropped
INFO:fontTools.subset:GPOS pruned
INFO:fontTools.subset:GSUB pruned
INFO:fontTools.subset:glyf pruned
INFO:fontTools.subset:Added gid0 to subset
INFO:fontTools.subset:Added first four glyphs to subset
INFO:fontTools.subset:Closing glyph list over 'GSUB': 5 glyphs before
INFO:fontTools.subset:Glyph names: ['.notdef', '.null', 'alpha', 'nonmarkingreturn', 'space']
INFO:fontTools.subset:Glyph IDs:   [0, 1, 2, 3, 837]
INFO:fontTools.subset:Closed glyph list over 'GSUB': 5 glyphs after
INFO:fontTools.subset:Glyph names: ['.notdef', '.null', 'alpha', 'nonmarkingreturn', 'space']
INFO:fontTools.subset:Glyph IDs:   [0, 1, 2, 3, 837]
INFO:fontTools.subset:Closing glyph list over 'glyf': 5 glyphs before
INFO:fontTools.subset:Glyph names: ['.notdef', '.null', 'alpha', 'nonmarkingreturn', 'space']
INFO:fontTools.subset:Glyph IDs

Saved: /Users/gabriel/Desktop/heavy_tails_abm/figures/ofc_demo_ccdf.pdf


## 6. OFC Phase Diagram

Grid search over `alpha_ofc ∈ [0.10, 0.24]` and `L ∈ {32, 64}`.  
Parameters kept small so the sweep runs in under 2 minutes.

In [ ]:
from experiments.ofc_phase_diagram import run_ofc_phase_diagram, plot_ofc_phase_diagram
b_means, b_stds = run_ofc_phase_diagram()
plot_ofc_phase_diagram(b_means, b_stds)

L=32 done
L=64 done
Saved: /Users/gabriel/Desktop/heavy_tails_abm/figures/ofc_phase_diagram.pdf


## 7. OFC Calibration on USGS Catalog

### 7a. MLE calibration

Find `alpha_ofc*` that minimises `|b_sim − b_emp|` where `b_emp` is the
Gutenberg-Richter b-value estimated from the USGS global catalog (M ≥ 2.0,
2000–2024).

In [ ]:
from experiments.ofc_calibration import run_mle_calibration, main as cal_main
from data.download_data import download_usgs_catalog
from utils.powerlaw_fit import gutenberg_richter_b
from experiments.ofc_calibration import run_mle_calibration, _ensure_figures_dir

_ensure_figures_dir()
catalog = download_usgs_catalog(min_magnitude=2.0, start="2000-01-01", end="2024-01-01")
magnitudes = catalog["magnitude"].dropna().values
magnitudes = magnitudes[magnitudes >= 4.5]
b_emp = gutenberg_richter_b(magnitudes, m_min=4.5)
print(f"Empirical b = {b_emp:.4f}")  # expected: approx 1.0
run_mle_calibration(b_emp)

INFO:data.download_data:Downloading USGS catalog: M>=2.0 from 2000-01-01 to 2024-01-01
INFO:data.download_data:  Fetching 2000-01-01 → 2000-04-01


Loading USGS catalog …


INFO:data.download_data:  Fetching 2000-04-01 → 2000-07-01
INFO:data.download_data:  Fetching 2000-07-01 → 2000-10-01
INFO:data.download_data:  Fetching 2000-10-01 → 2001-01-01
INFO:data.download_data:  Fetching 2001-01-01 → 2001-04-01
INFO:data.download_data:  Fetching 2001-04-01 → 2001-07-01
INFO:data.download_data:  Fetching 2001-07-01 → 2001-10-01
INFO:data.download_data:  Fetching 2001-10-01 → 2002-01-01
INFO:data.download_data:  Fetching 2002-01-01 → 2002-04-01
INFO:data.download_data:  Fetching 2002-04-01 → 2002-07-01
INFO:data.download_data:  Fetching 2002-07-01 → 2002-10-01
INFO:data.download_data:  Fetching 2002-10-01 → 2003-01-01
INFO:data.download_data:  Fetching 2003-01-01 → 2003-04-01
INFO:data.download_data:  Fetching 2003-04-01 → 2003-07-01
INFO:data.download_data:  Fetching 2003-07-01 → 2003-10-01
INFO:data.download_data:  Fetching 2003-10-01 → 2004-01-01
INFO:data.download_data:  Fetching 2004-01-01 → 2004-04-01
INFO:data.download_data:  Fetching 2004-04-01 → 2004-07-

869,692 events, M in [0.7, 9.1]
Empirical b = 0.1652
  alpha=0.100 → b_sim=3.1713
  alpha=0.116 → b_sim=2.6724
  alpha=0.131 → b_sim=2.3257
  alpha=0.147 → b_sim=2.0512
  alpha=0.162 → b_sim=1.8803
  alpha=0.178 → b_sim=1.7034
  alpha=0.193 → b_sim=1.6279
  alpha=0.209 → b_sim=1.5815
  alpha=0.224 → b_sim=1.5940
  alpha=0.240 → b_sim=0.6767

alpha_ofc* = 0.2400  (b_sim=0.6767, b_emp=0.1652)


INFO:fontTools.subset:maxp pruned
INFO:fontTools.subset:cmap pruned
INFO:fontTools.subset:kern dropped
INFO:fontTools.subset:post pruned
INFO:fontTools.subset:FFTM dropped
INFO:fontTools.subset:GPOS pruned
INFO:fontTools.subset:GSUB pruned
INFO:fontTools.subset:glyf pruned
INFO:fontTools.subset:Added gid0 to subset
INFO:fontTools.subset:Added first four glyphs to subset
INFO:fontTools.subset:Closing glyph list over 'GSUB': 5 glyphs before
INFO:fontTools.subset:Glyph names: ['.notdef', '.null', 'alpha', 'nonmarkingreturn', 'space']
INFO:fontTools.subset:Glyph IDs:   [0, 1, 2, 3, 837]
INFO:fontTools.subset:Closed glyph list over 'GSUB': 5 glyphs after
INFO:fontTools.subset:Glyph names: ['.notdef', '.null', 'alpha', 'nonmarkingreturn', 'space']
INFO:fontTools.subset:Glyph IDs:   [0, 1, 2, 3, 837]
INFO:fontTools.subset:Closing glyph list over 'glyf': 5 glyphs before
INFO:fontTools.subset:Glyph names: ['.notdef', '.null', 'alpha', 'nonmarkingreturn', 'space']
INFO:fontTools.subset:Glyph IDs

Saved: /Users/gabriel/Desktop/heavy_tails_abm/figures/ofc_calibration_mle.pdf


### 7b. ABC calibration (skipped in notebook)

ABC-SMC calibration is **not run here** because it requires O(hours) of
compute (`N_SEEDS=20`, `N_EVENTS=50_000`, 5 SMC populations × 100
particles).

To run it separately:

```bash
python experiments/ofc_calibration.py
```

This produces `figures/ofc_calibration_abc.pdf` with the posterior
distribution over `alpha_ofc`.